In [ ]:
# ==========================================
# 1. Environment & Imports
# ==========================================
import json
import torch
import numpy as np
from transformers import AutoProcessor, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer
from sentence_transformers import SentenceTransformer, util

# Load the Sentence-BERT model for the R_bonus qualitative reward
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# ==========================================
# 2. The Reward Functions (Section 3.2.2 Logic)
# ==========================================

def format_and_schema_penalty(completions, **kwargs):
    """
    P_schema: Structural Penalty
    Calculates \lambda_1 * v_syntax + \lambda_2 * v_schema
    Returns a strong negative reward if the JSON is broken or missing critical keys.
    """
    rewards = []
    for comp in completions:
        try:
            # 1. Syntax Check (v_syntax)
            spec = json.loads(comp)
            
            # 2. Schema Check (v_schema)
            # Mapping to your SpecExtraction.py keys: topo, axes, ser, rel
            required_keys = {"topo", "axes", "ser", "rel"}
            missing_keys = required_keys - set(spec.keys())
            
            if missing_keys:
                rewards.append(-0.5 * len(missing_keys)) # Penalty for missing keys
            else:
                rewards.append(0.5) # Baseline positive for perfect syntax and schema
        except json.JSONDecodeError:
            rewards.append(-2.0) # Massive penalty for broken JSON syntax
    return rewards

def spec_align_reward(completions, ground_truth, **kwargs):
    """
    R_align (Weight: 0.6)
    Computes field-wise accuracy for topology, axes, and series data.
    """
    rewards = []
    for comp, gt_str in zip(completions, ground_truth):
        try:
            pred = json.loads(comp)
            gt = json.loads(gt_str)
            score = 0.0
            total_checks = 0
            
            # 1. Topology Match (Exact)
            if pred.get("topo", {}).get("type") == gt.get("topo", {}).get("type"):
                score += 1.0
            total_checks += 1
            
            # 2. Data Point Bounding (Numerical Tolerance)
            # Simplified check: did it extract the correct number of series?
            pred_ser = pred.get("ser", [])
            gt_ser = gt.get("ser", [])
            if len(pred_ser) == len(gt_ser) and len(gt_ser) > 0:
                score += 1.0
            total_checks += 1
                
            # Normalize and apply the 0.6 weight
            normalized_score = (score / total_checks) * 0.6
            rewards.append(normalized_score)
            
        except:
            rewards.append(0.0) # If it fails to parse, it gets 0 alignment reward
    return rewards

def semantic_trend_reward(completions, ground_truth, **kwargs):
    """
    R_trend (Weight: 0.4)
    Evaluates the reasoning logic: stats, extrema, and global_trend.
    """
    rewards = []
    for comp, gt_str in zip(completions, ground_truth):
        try:
            pred = json.loads(comp)
            gt = json.loads(gt_str)
            score = 0.0
            total_checks = 0
            
            pred_ser = pred.get("ser", [])
            gt_ser = gt.get("ser", [])
            
            for p_s, g_s in zip(pred_ser, gt_ser):
                # A. Global Trend Check
                p_trend = p_s.get("trend", {}).get("global_trend")
                g_trend = g_s.get("trend", {}).get("global_trend")
                if g_trend:
                    if p_trend == g_trend: score += 1.0
                    total_checks += 1
                
                # B. Extrema Check (using SpecExtraction's 'stats' key)
                p_max = p_s.get("stats", {}).get("max_value")
                g_max = g_s.get("stats", {}).get("max_value")
                if g_max and p_max:
                    # Allow slight floating point variation
                    if abs(float(p_max) - float(g_max)) < 0.5: score += 1.0
                    total_checks += 1

            if total_checks == 0:
                rewards.append(0.0)
            else:
                rewards.append((score / total_checks) * 0.4)
        except:
            rewards.append(0.0)
    return rewards

def qualitative_bonus_reward(completions, ground_truth, **kwargs):
    """
    R_bonus: Sentence-BERT Cosine Similarity
    Evaluates the 'rel' (comparative_relations) text spans.
    """
    rewards = []
    for comp, gt_str in zip(completions, ground_truth):
        try:
            pred = json.loads(comp)
            gt = json.loads(gt_str)
            
            # Extract the relation descriptions from the 'rel' array
            pred_rels = " ".join([r.get("relation", "") for r in pred.get("rel", [])])
            gt_rels = " ".join([r.get("relation", "") for r in gt.get("rel", [])])
            
            if not gt_rels:
                rewards.append(0.0)
                continue
                
            # Compute embeddings and cosine similarity
            emb_pred = sbert_model.encode(pred_rels, convert_to_tensor=True)
            emb_gt = sbert_model.encode(gt_rels, convert_to_tensor=True)
            cosine_score = util.pytorch_cos_sim(emb_pred, emb_gt).item()
            
            # Apply R_bonus = +0.1 if similarity >= 0.8
            if cosine_score >= 0.8:
                rewards.append(0.1)
            else:
                rewards.append(0.0)
        except:
            rewards.append(0.0)
    return rewards


In [ ]:

# ==========================================
# 3. Model & GRPO Trainer Setup
# ==========================================

# 1. Load the SFT Model (You MUST start from your SFT checkpoint, not the base model)
model_id = "/workspace/qwen_lora_full_run" 

print("Loading SFT Model for RL...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

# 2. Configure GRPO
grpo_config = GRPOConfig(
    output_dir="/workspace/qwen_grpo_stage1",
    learning_rate=1e-5,              # Must be much lower than SFT (e.g., 1e-5)
    lr_scheduler_type="cosine",
    per_device_train_batch_size=1,   # 1 chart per GPU
    num_generations=4,               # The "Group" size (4 candidate JSONs per chart)
    max_completion_length=4096,      # Match your SFT length
    gradient_accumulation_steps=4,
    beta=0.04,                       # KL divergence penalty (prevents model from breaking SFT logic)
    logging_steps=10,
    save_steps=50,
    bf16=True,
)

# 3. Initialize Trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=processor,
    reward_funcs=[
        format_and_schema_penalty, 
        spec_align_reward, 
        semantic_trend_reward, 
        qualitative_bonus_reward
    ],
    args=grpo_config,
    train_dataset=grpo_dataset, # Your formatted (image, text) dataset
)

# 4. Launch RL Rollouts
print("Starting GRPO Rollouts...")
trainer.train()